In [ ]:
import os 
from IPython.display import display
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell  
from ipywidgets import interact  
InteractiveShell.ast_node_interactivity = "all"   # type: ignore
import ipywidgets as widgets
from ipywidgets import interact 
from pandas import Index
from typing import Any, TypedDict
import torch
from src.base.lit_module.components.metrics_standard import MyMultiClassAccuracy

In [ ]:
""" 
Required classes. 
"""

class MetricMetadata(TypedDict):
    epochs: list[float]
    steps: list[float]

class PerStepMetricInfo(TypedDict):
    values: np.ndarray
    save_path: Path
    header: list[str]

In [ ]:
"""
Interactive analysis functionality.
"""

# def show_col_summary(column: Any, frame):
#     print(frame[column].describe())

# def show_summary(dfs: list[pd.DataFrame], v_dir: Path):
#     print(f"Results aggregated over {v_dir.stem} in {v_dir.parent.parent.stem}/{v_dir.parent.stem}")
#     for frame in dfs:
#         interact(show_col_summary, frame=frame, column=list(frame.columns))
#         frame 


In [ ]:
""" 
Visualization functionality. 
"""

def viz_sep_col(metric: PerStepMetricInfo, save_fig: bool = True, save_derived_metric: bool = False) -> None:
    """
    For a metric object instance, visualize each col of `values` with its associated string in `header`.
    Internal output is figure with plots per column, header label combo.
    Optionally save each to compressed numpy file with flags (NOT recommended).
    Optionally save figure (Recommended). 
    TODO: But with what path, might as well maybe create separate instance separate from global df, not really. 
    """
    n_cols = metric["values"].shape[1]
    fig, axs = plt.subplots(1, n_cols, sharex=True, sharey=True, figsize=(4 * n_cols, 4))
    fig.subplots_adjust(wspace=0, hspace=0)

    for i, _ in enumerate(metric["header"]):
        col_vals = metric["values"][:,i]
        col_name = metric["header"][i]
        clean_header = col_name.replace('.', '_')
        col_path = Path(metric['save_path'].parent.resolve() / clean_header)
        axs[i].plot(
            col_vals, 
            # linestyle="dashed", 
            # color="black", 
            # marker="o", 
            label=clean_header
        )
        axs[i].legend()

        if save_derived_metric:
            if not Path.exists(col_path):
                col_path.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(file=col_path.resolve(), derived_metric = col_vals)

    if save_fig: 
        save_path = Path(str(metric['save_path'].resolve()) + ".png")
        fig.savefig(save_path, format="png")
        
def viz_all_col(metric_df: pd.DataFrame, unified_name: str = "Train Accuracy") -> None:
    """
    For a metric object instance, visualize all col of `values` with its associated string in `header`.
    Internal output is figure with 1 plot for all column , labelled by col name in Index.
    Optionally save as compressed numpy file with flags per viz (Recommended).
    Optionally save figure (Recommended) per viz. 
    """
    n_cols = len(metric_df.columns)
    fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(n_cols, n_cols))
    # for col in metric_df.columns:
    #     metric_df.plot(sharex=True, sharey=True, ax=ax, y=col, legend=True)
    metric_df.plot()
    ax.set_title(unified_name)
    ax.set_xticklabels(list(metric_df.index))
    ax.set_ylabel(unified_name)
    ax.legend()


In [ ]:
"""
Standard metric analysis helpers. 
"""
def summarize_run_acc(run_metrics_paths: list[Path], prefix:str):
    """ 
    This summarises accuracy metrics across a run.
    Output is in the parent directory of the current path.  
    """
    enumerated_metrics = enumerate(run_metrics_paths)

    run_dfs:list[pd.DataFrame] = []
    for idx, train_metrics_path in enumerated_metrics:
        metric = torch.load(train_metrics_path.resolve(), map_location="cpu")
        classes = list(metric.keys())
        base_acc = MyMultiClassAccuracy(classes)
        base_acc.load_state_dict(metric)
        print(type(base_acc))

        # Convert tensors not addressed by `.compute()`
        clean_epochs = [tensor.item() for tensor in base_acc.metric_state["epochs"]]
        clean_steps = [tensor.item() for tensor in base_acc.metric_state["steps"]]

        metadata = MetricMetadata(epochs=clean_epochs, steps=clean_steps) # Unify values in new dict
        metadata_df = pd.DataFrame(metadata)
        head = metadata_df.head(10)
        index = pd.MultiIndex.from_frame(metadata_df)

        computed = base_acc.compute()

        data_df = pd.DataFrame(computed, index=index).replace(100, np.nan)
        run_dfs.append(data_df)

    df_all = pd.concat(run_dfs)
    mean_df = df_all.groupby(level=0).mean(skipna=True)
    std_df = df_all.groupby(level=0).std(skipna=True)
    summary_mean = mean_df.describe()
    summary_std = std_df.describe() 
    display(summary_mean)
    display(summary_std)
    viz_all_col(mean_df, unified_name="Train Accuracy Mean")    
    viz_all_col(std_df, unified_name="Train Accuracy Std")
    plt.show()
    print("shown")    

In [ ]:
""" 
GCS analysis helpers. 
"""
def analyse_cs_metrics(metrics_paths: list[Path], prefix:str) -> None:
    """ 
    Perform analysis steps of correlation store metrics. 
    """
    summarize_run_acc(metrics_paths, prefix)

In [ ]:
outputs_dir = Path().resolve() / "outputs"
corr_store_names = dict(normed="GCS", raw="RGCS")

In [ ]:
""" 
Main entry point for analysis. 
TODO: 
- Fix FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\motlh\\coder_life\\COS700\\outputs\\adam_class_imbalance_ratios\\multijob\\v_0\\run_0\\train\\gcs_global_std\\global_stdconv1.weight.npz'
via change .weight. This is in viz_sep_col, trying for first time. 
- Merge analyze like with analyze, to not duplicate code changes. Can try decorator maybe.
- Change to global/std/"<old>" naming for graphs. 
- Add cleaning routine (redundant if training works fine and I know consistent output directories)
    - So cleaning is just for current mistakes of files added, or simply retrain. 
- Debug save as png does not work unless explicit. 
- Save GCS as npz with metadata e.g. step count, header, stitched after all epochs, not saved per epoch, just like normal metrics, as flat file. 
- Save metrics as npz with meta data e.g. step count with callbacks likely. 
- Run to check efficiency, here then cluster. 
"""

plt.switch_backend('WebAgg')

for experim_dir in sorted(outputs_dir.iterdir()):
    version_dirs = sorted(experim_dir.glob("multijob/R*"))

    for v_dir in version_dirs:
        # Get all csv file path matches to default logger file name for standard metrics
        train_acc_paths = sorted(v_dir.rglob("run*/train_acc.pt", case_sensitive=False))
        if len(train_acc_paths) > 0:
            summarize_run_acc(train_acc_paths, prefix=train_acc_paths[0].stem)
    
        val_acc_paths = sorted(v_dir.rglob("run*/val_acc.pt", case_sensitive=False))
        if len(val_acc_paths) > 0:
            summarize_run_acc(val_acc_paths, prefix=val_acc_paths[0].stem)

        # Get all csv file path matches to custom name for both cs metrics
        # gcs_paths = sorted(v_dir.rglob(f"run*/train/*{corr_store_names["normed"]}*.pt", case_sensitive=False))
        # analyse_cs_metrics(gcs_paths, "gcs")

        # rgcs_paths = sorted(v_dir.rglob(f"run*/train/*{corr_store_names["raw"]}*.pt", case_sensitive=False))
        # analyse_cs_metrics(rgcs_paths, "rgcs")